# Data cleaning & Preprocessing

> TODO adicionar descrição breve sobre o que fiz

## Data cleaning

In [20]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import missingno as msno
sns.set_theme(style='darkgrid')
import matplotlib.pyplot as plt

In [21]:
data = pd.read_excel('Assignment/DataSets/UCMF.xls')

data = data.rename(columns={
    'IDADE': 'Idade',
    'PULSOS':'Pulsos',
    'PA SISTOLICA': 'SBP',
    'PA DIASTOLICA': 'DBP',
    'PPA': 'Result dbp-DBP',
    'NORMAL X ANORMAL': 'Patologia',
    'SOPRO' : 'Sopro',
    'SEXO':'Sexo',
    'HDA 1': 'HDA1',
    'MOTIVO1' : 'Motivo1',
    'MOTIVO2': 'Motivo2'
})

In [22]:
data.columns

Index(['ID', 'Peso', 'Altura', 'IMC', 'Atendimento', 'DN', 'Idade', 'Convenio',
       'Pulsos', 'SBP', 'DBP', 'Result dbp-DBP', 'Patologia', 'B2', 'Sopro',
       'FC', 'HDA1', 'HDA2', 'Sexo', 'Motivo1', 'Motivo2'],
      dtype='object')

### Remove Irrelevant features

In [23]:
data = data.drop(['ID','Atendimento','DN','Convenio'],axis=1)

In [24]:
data.columns

Index(['Peso', 'Altura', 'IMC', 'Idade', 'Pulsos', 'SBP', 'DBP',
       'Result dbp-DBP', 'Patologia', 'B2', 'Sopro', 'FC', 'HDA1', 'HDA2',
       'Sexo', 'Motivo1', 'Motivo2'],
      dtype='object')

### Filter records with desired target group age (2-19)

In [25]:
data.shape

(17873, 17)

In [26]:

data = data[data['Idade'] != '#!VALUE!']
data = data[data['Idade'] != '#VALUE!']

In [27]:
data.shape

(17753, 17)

In [28]:
data['Idade'] = data['Idade'].astype(float)


In [29]:
data = data[data['Idade'].between(2,19)]
data.shape

(12375, 17)

In [30]:
data['Idade'].value_counts

<bound method IndexOpsMixin.value_counts of 4         9.60
5         4.40
6        12.89
7         5.89
10        6.24
         ...  
17862    12.30
17863     8.30
17865     8.67
17867     2.14
17872    11.38
Name: Idade, Length: 12375, dtype: float64>

### Handle Inconsistent values

In [31]:
# Patologia
print("(Before)", data['Patologia'].value_counts())
map = {
    'anormal': 'Anormal',
    'Normais': 'Normal'
}
data['Patologia'] = data['Patologia'].replace(map)
print("(After)", data['Patologia'].value_counts())



(Before) Patologia
Normal     7764
Anormal    4310
anormal       1
Normais       1
Name: count, dtype: int64
(After) Patologia
Normal     7765
Anormal    4311
Name: count, dtype: int64


In [32]:
# Sexo
print("(Before)",data['Sexo'].value_counts())
data['Sexo'] = data['Sexo'].str.strip().str.capitalize()
map = {
    'M': 'Masculino',
    'F': 'Feminino'
}
data['Sexo'] = data['Sexo'].replace(map)
print("(After)", data['Sexo'].value_counts())

(Before) Sexo
M                6641
F                4882
Masculino         463
Feminino          178
Indeterminado     146
masculino          62
Name: count, dtype: int64
(After) Sexo
Masculino        7166
Feminino         5060
Indeterminado     146
Name: count, dtype: int64


In [33]:
# Pulsos
print("(Before)",data['Pulsos'].value_counts())
data['Pulsos'] = data['Pulsos'].str.strip().str.capitalize()
map = {
    'Normais': 'Normal',
    'NORMAIS': 'Normal',
    'Diminuídos': 'Femorais diminuidos',
}
data['Pulsos'] = data['Pulsos'].replace(map)
print("(After)", data['Pulsos'].value_counts())


(Before) Pulsos
Normais                12010
Outro                     26
Amplos                    17
Femorais diminuidos        9
Diminuídos                 7
NORMAIS                    1
Name: count, dtype: int64
(After) Pulsos
Normal                 12011
Outro                     26
Amplos                    17
Femorais diminuidos       16
Name: count, dtype: int64


In [34]:
# B2
print("(Before)",data['B2'].value_counts())
data['B2'] = data['B2'].str.strip().str.capitalize()
map = {
    'Única': 'Unico',
    'Desdob fixo': 'Split fixo'
}
data['B2'] = data['B2'].replace(map)
print("(After)", data['B2'].value_counts())

(Before) B2
Normal           11720
Hiperfonética      136
Desdob fixo        115
Outro               79
Única               28
Name: count, dtype: int64
(After) B2
Normal           11720
Hiperfonética      136
Split fixo         115
Outro               79
Unico               28
Name: count, dtype: int64


In [35]:
# Sopro
print("(Before)",data['Sopro'].value_counts())
data['Sopro'] = data['Sopro'].str.strip().str.capitalize()
map = {
    'ausente': 'Ausente',
    'sistólico': 'Sistólico',
    'contínuo': 'Contínuo',
    'diastólico': 'Diastólico'
}
data['Sopro'] = data['Sopro'].replace(map)
print("(After)", data['Sopro'].value_counts())

(Before) Sopro
ausente                   8254
Sistólico                 3019
sistólico                  771
contínuo                    16
Contínuo                    13
diastólico                   8
Sistolico e diastólico       2
Name: count, dtype: int64
(After) Sopro
Ausente                   8254
Sistólico                 3790
Contínuo                    29
Diastólico                   8
Sistolico e diastólico       2
Name: count, dtype: int64


In [36]:
# Motivo1
data['Motivo1'].value_counts()
print("(Before)",data['Motivo1'].value_counts())
map = {
    '5 - Parecer cardiológico' : 'Triagem Cardiológica',
    '6 - Suspeita de cardiopatia': 'Possivel cardiopatia',
    '2 - Check-up' : 'Checkup de rotina',
    '1 - Cardiopatia já estabelecida': 'Cardiopatia',
    '7 - Outro' : 'Outro'
}
data['Motivo1'] = data['Motivo1'].replace(map)
print("(After)", data['Motivo1'].value_counts())

(Before) Motivo1
5 - Parecer cardiológico           6470
6 - Suspeita de cardiopatia        3911
2 - Check-up                        788
1 - Cardiopatia já estabelecida     784
7 - Outro                           314
Name: count, dtype: int64
(After) Motivo1
Triagem Cardiológica    6470
Possivel cardiopatia    3911
Checkup de rotina        788
Cardiopatia              784
Outro                    314
Name: count, dtype: int64


In [37]:
# Motivo2
data['Motivo2'].value_counts()
print("(Before)",data['Motivo2'].value_counts())
map = {
    '5 - Cirurgia': 'Cirurgia',
    '6 - Sopro' : 'Presença de Sopro',
    '5 - Atividade física' : 'Atividade física',
    'Outro' : 'Outros',
    '1 - Cardiopatia congenica' : 'Cardiopatia congenica',
    '6 - Dor precordial' : 'Outros',
    '6 - Palpitação/taquicardia/arritmia' : 'Outros',
    '6 - HAS/dislipidemia/obesidade' : 'Fatores de risco',
    '6 - Dispnéia' : 'Outros',
    '1 - Cardiopatia adquirida' :'Outros',
    '6 - Cianose' : 'Outros',
    '6 - Cardiopatia na familia' : 'Fatores de risco',
    '6 - Cansaço' : 'Outros',
    '6 - Alterações de pulso/perfusão' : 'Outros',
    '6 - Cianose e dispnéia' : 'Outros',
    '5 - Uso de cisaprida' : 'Outros'

}
data['Motivo2'] = data['Motivo2'].replace(map)
print("(After)", data['Motivo2'].value_counts())


(Before) Motivo2
5 - Cirurgia                           3351
6 - Sopro                              1718
5 - Atividade física                   1069
Outro                                   719
1 - Cardiopatia congenica               639
6 - Dor precordial                      615
6 - Palpitação/taquicardia/arritmia     452
6 - HAS/dislipidemia/obesidade          395
6 - Dispnéia                            240
1 - Cardiopatia adquirida               112
6 - Cianose                              51
6 - Cardiopatia na familia               31
6 - Cansaço                              16
6 - Alterações de pulso/perfusão          2
6 - Cianose e dispnéia                    2
5 - Uso de cisaprida                      1
Name: count, dtype: int64
(After) Motivo2
Cirurgia                 3351
Outros                   2210
Presença de Sopro        1718
Atividade física         1069
Cardiopatia congenica     639
Fatores de risco          426
Name: count, dtype: int64


### Remove incorect/impossible/irrelevant values

In [38]:
# Remover objectos com valores incorrectos de SBP (da-mos uma 'folga' de 10 para
# limite superior e inferior)
import json
with open('Assignment/Data-structures/sbp-dbp-reference-values-map.json', 'r') as file:
    sbp_map = json.load(file)

print(f'(Before removing incorret Male/Female SBP Values) {data.shape} ')

# Remover valores SBP invalidos para sexo Masculino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Masculino'].iterrows():
    idade = str(int(row['Idade']))
    sbp = row['SBP']
    lower_limit = float(sbp_map['M'][idade]['SBP'][0] - 10)
    upper_limit = float(sbp_map['M'][idade]['SBP'][1] + 10)
    if (sbp < lower_limit or sbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)
print(f'Male entries removed {len(indices_to_drop)}')

# Remover valores SBP invalidos para sexo Feminino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Feminino'].iterrows():
    idade = str(int(row['Idade']))
    sbp = row['SBP']
    lower_limit = float(sbp_map['F'][idade]['SBP'][0] - 10)
    upper_limit = float(sbp_map['F'][idade]['SBP'][1] + 10)
    if (sbp < lower_limit or sbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)

print(f'Female Entries removed {len(indices_to_drop)}')
print(f'(After removing incorret Male/Female SBP Values) {data.shape} ')

(Before removing incorret Male/Female SBP Values) (12375, 17) 
Male entries removed 206
Female Entries removed 142
(After removing incorret Male/Female SBP Values) (12027, 17) 


In [39]:
# Remover objectos com valores incorrectos de DBP (da-mos uma 'folga' de 10 para
# limite superior e inferior)
import json
with open('Assignment/Data-structures/sbp-dbp-reference-values-map.json', 'r') as file:
    dbp_map = json.load(file)

print(f'(Before removing incorret Male/Female DBP Values) {data.shape} ')

# Remover valores DBP invalidos para sexo Masculino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Masculino'].iterrows():
    idade = str(int(row['Idade']))
    dbp = row['DBP']
    lower_limit = float(dbp_map['M'][idade]['DBP'][0] - 10)
    upper_limit = float(dbp_map['M'][idade]['DBP'][1] + 10)
    if (dbp < lower_limit or dbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)
print(f'Male entries removed {len(indices_to_drop)}')

# Remover valores DBP invalidos para sexo Feminino
indices_to_drop = []
for index, row in data[data['Sexo'] == 'Feminino'].iterrows():
    idade = str(int(row['Idade']))
    dbp = row['DBP']
    lower_limit = float(dbp_map['F'][idade]['DBP'][0] - 10)
    upper_limit = float(dbp_map['F'][idade]['DBP'][1] + 10)
    if (dbp < lower_limit or dbp > upper_limit):
        indices_to_drop.append(index)
        
data = data.drop(indices_to_drop)

print(f'Female Entries removed {len(indices_to_drop)}')
print(f'(After removing incorret Male/Female DBP Values) {data.shape} ')

(Before removing incorret Male/Female DBP Values) (12027, 17) 
Male entries removed 19
Female Entries removed 29
(After removing incorret Male/Female DBP Values) (11979, 17) 


In [40]:
# Remover objectos com valores imposiveis de peso 
print(f'Before removing impossible Weight values {data.shape}')
data = data.drop(data[data['Peso'] <= 0].index)
print(f'After removing impossible Weight values {data.shape}')


Before removing impossible Weight values (11979, 17)
After removing impossible Weight values (10753, 17)


In [41]:
# Remover objectos com valores imposiveis de Altura 
print(f'Before removing impossible Heigth values {data.shape}')
data = data.drop(data[data['Altura'] <=0].index)
print(f'After removing impossible Heigth values {data.shape}')



Before removing impossible Heigth values (10753, 17)
After removing impossible Heigth values (9548, 17)


In [42]:
# Remover Pulsos = Outro
print(f'Before removing irrelevant Pulsos values {data.shape}')
data = data.drop(data[data['Pulsos'] == 'Outro'].index)
print(f'After removing irrelevant Pulsos values {data.shape}')


Before removing irrelevant Pulsos values (9548, 17)
After removing irrelevant Pulsos values (9535, 17)


> Removed objects with ```Pulsos = Outro```, this value gives no info about the Pulse related to the object, so we removed because we think this objects  will not help our prediction task

In [43]:
# Remover valores B2 = Outro
print(f'Before removing irrelevant B2 values {data.shape}')
data = data.drop(data[data['B2'] == 'Outro'].index)
print(f'After removing irrelevant B2 values {data.shape}')


Before removing irrelevant B2 values (9535, 17)
After removing irrelevant B2 values (9477, 17)


> Removed objects with ```B2 = Outro```, because this value gives no info about the type of the second heart sound related to the object, so we removed because we think this objects will not help our prediction task

In [44]:
# Remover valores Sopro = Sistolico e diastólico 
print(f'Before removing irrelevant Sopro values {data.shape}')
data = data.drop(data[data['Sopro'] == 'Sistolico e diastólico'].index)
print(f'After removing irrelevant Sopro values {data.shape}')

Before removing irrelevant Sopro values (9477, 17)
After removing irrelevant Sopro values (9476, 17)


> Removed objects with ```Sopro = Sistolico e diastólico```, because it just one object with this value, and we think it will not help our prediction task

In [ ]:
# Ajustar unico valor HDA2 = Assintomático e passar para HDA1 = Assintomático
print(data['HDA2'].value_counts())

data.loc[data['HDA2'] == 'Assintomático', 'HDA1'] = 'Assintomático'
data.loc[data['HDA2'] == 'Assintomático', 'HDA2'] = np.nan
print(data['HDA2'].value_counts())

HDA2
Palpitacao         99
Dispneia           73
Dor precordial     67
Desmaio/tontura    48
Outro              36
Cianose            33
Ganho de peso      26
Assintomático       1
Name: count, dtype: int64
HDA2
Palpitacao         99
Dispneia           73
Dor precordial     67
Desmaio/tontura    48
Outro              36
Cianose            33
Ganho de peso      26
Name: count, dtype: int64


In [56]:
# Remover valores HDA1 = Outro e HDA2 = Outro
print(f'Before removing irrelevant HDA1 and HDA2 values {data.shape}')
data = data.drop(data[data['HDA1'] == 'Outro'].index)
data = data.drop(data[data['HDA2'] == 'Outro'].index)
print(f'After removing irrelevant HDA1 and HDA2 values {data.shape}')


Before removing irrelevant HDA1 and HDA2 values (9476, 17)
After removing irrelevant HDA1 and HDA2 values (9319, 17)


> Removed objects with ```HDA1 = Outro``` and ```HDA2 = Outro```, because the information about history of disease it provides is to ambiguos. Having this in mind, we think this objects will not help de prediction task, so we removed them

In [59]:
# Remover valores Sexo = Indeterminado 
print(f'Before removing irrelevant Sexo values {data.shape}')
data = data.drop(data[data['Sexo'] == 'Indeterminado'].index)
print(f'After removing irrelevant Sexo values {data.shape}')

Before removing irrelevant Sexo values (9319, 17)
After removing irrelevant Sexo values (9210, 17)


> Removed objects with ```Sexo = Indeterminado```, because it does not give any info about the patient sex, therefore it does not bring any value to the prediction task 

In [ ]:
# Ajustar coluna FC para ser numérica 
# Tenta converter para número. O que tiver letras vira NaN (errors='coerce')
data['FC'] = pd.to_numeric(data['FC'], errors='coerce')

In [ ]:
# Remover objetos de Frequencia impossiveis
import json
with open('Assignment/Data-structures/fc-reference-values.json', 'r') as file:
    fc_map = json.load(file)

json.dumps(fc_map)

print(f'(Before removing invalid FC Values) {data.shape} ')

# Remover valores SBP invalidos para sexo Feminino
indices_to_drop = []
for index, row in data.iterrows():
    card_freq = row['FC']

    idade = str(int(row['Idade']))
    min_freq = fc_map[idade]['bpm_minimo'] - 5.0
    max_freq = fc_map[idade]['bpm_maximo'] + 5.0
    if (~np.isnan(card_freq)):
        if(card_freq < min_freq or card_freq > max_freq):
            indices_to_drop.append(index)

data = data.drop(indices_to_drop)
print(f'(After removing invalid FC Values) {data.shape} ')

(Before removing invalid FC Values) (9210, 17) 
72.0
120.0
140.0
150.0
10.0
130.0
70.0
112.0
72.0
72.0
72.0
130.0
50.0
70.0
120.0
50.0
150.0
72.0
55.0
72.0
72.0
72.0
120.0
130.0
64.0
72.0
72.0
72.0
128.0
72.0
72.0
64.0
50.0
72.0
140.0
72.0
72.0
150.0
120.0
140.0
140.0
70.0
70.0
50.0
72.0
150.0
72.0
8096.0
110.0
52.0
116.0
70.0
150.0
72.0
72.0
60.0
72.0
60.0
72.0
120.0
110.0
900.0
72.0
72.0
110.0
72.0
72.0
180.0
110.0
72.0
128.0
70.0
150.0
116.0
110.0
148.0
110.0
50.0
72.0
110.0
60.0
60.0
72.0
72.0
57.0
160.0
801.0
110.0
128.0
150.0
50.0
140.0
120.0
72.0
72.0
120.0
110.0
230.0
68.0
120.0
68.0
68.0
112.0
72.0
60.0
70.0
110.0
68.0
72.0
110.0
64.0
70.0
64.0
140.0
112.0
110.0
108.0
132.0
72.0
110.0
72.0
60.0
16.0
130.0
146.0
112.0
110.0
130.0
68.0
50.0
108.0
60.0
60.0
68.0
72.0
52.0
132.0
72.0
110.0
148.0
64.0
68.0
68.0
72.0
110.0
60.0
110.0
72.0
72.0
72.0
160.0
72.0
70.0
120.0
120.0
48.0
48.0
72.0
72.0
140.0
60.0
74.0
110.0
130.0
110.0
140.0
72.0
140.0
180.0
130.0
110.0
110.0
72.0
120.0
12

TODO:
1) Corrigir Frequencia cardiacas (valores que não fazem sentido, e secalhar definir intervalos de frequencia)



### TODO

1) Colocar os valores de SBP e DBP como Normal, Limite e Hipertenso (de acordo com os valores de referencia)
2) Recalcular Result SBP-PPA
3) Recalcular BMI (remover os passos do valor de refrencia)
4) Alterar Idade para ser categorica (baseado no intervalo)
  

> Usar o document.pdf como referência

## Data Preprocessing